# ✈️ Travel Agent Chatbot
**Built with LangChain + Gemini | Google Colab**

A streaming travel-only chatbot with manual list-based memory and auto-summarization every 5 turns.

---
## 📋 Table of Contents
1. [Install Dependencies](#cell-install)
2. [Configuration & API Key](#cell-config)
3. [Prompt Templates](#cell-prompts)
4. [Memory Manager](#cell-memory)
5. [Streaming & Chat Engine](#cell-engine)
6. [Interactive Chatbot Demo](#cell-demo)

---
### 🏗️ Architecture
```
User Input
    │
    ▼
┌─────────────────────────────────┐
│  Travel Guard (prompt check)    │  ← blocks non-travel questions
└──────────────┬──────────────────┘
               │ travel only
    ▼
┌─────────────────────────────────┐
│  Memory Manager (Python List)   │  ← stores all messages
│  • Appends user + AI messages   │
│  • Summarizes every 5 turns     │
│  • Replaces history with summary│
└──────────────┬──────────────────┘
               │
    ▼
┌─────────────────────────────────┐
│  LangChain Chain (Gemini LLM)   │  ← streamed token-by-token
└─────────────────────────────────┘
```

---
## ⚙️ Cell 1 — Install Dependencies

In [ ]:
# ============================================================
# CELL 1: Install Dependencies
# ============================================================
# After this cell finishes:
#   Runtime → Restart runtime → then run from Cell 2 onward

import sys, subprocess

PACKAGES = [
    "langchain>=0.2.0",
    "langchain-google-genai",   # Gemini via LangChain
    "langchain-core>=0.2.0",
    "google-genai",             # Google GenAI SDK (new)
    "rich>=13.0.0",             # pretty terminal output
]

subprocess.check_call([sys.executable, "-m", "pip", "install", "-q"] + PACKAGES)

print("\u2705 All packages installed.")
print("\u26a0\ufe0f  Now: Runtime \u2192 Restart runtime, then run from Cell 2.")

---
## 🔑 Cell 2 — Configuration & API Key

In [ ]:
# ============================================================
# CELL 2: Configuration
# ============================================================
# Get your free API key: https://aistudio.google.com/app/apikey

import os, getpass

# ── Model settings ───────────────────────────────────────
MODEL_NAME    = "gemini-2.0-flash-lite"  # fast, free-tier friendly
TEMPERATURE   = 0.7                       # some creativity for travel tips
STREAM        = True                      # token-by-token output
SUMMARY_EVERY = 5                         # summarize after every N turns

# ── API key ──────────────────────────────────────────────
if not os.environ.get("GOOGLE_API_KEY"):
    os.environ["GOOGLE_API_KEY"] = getpass.getpass(
        "\U0001f511 Enter your Google AI Studio API key: "
    )

print(f"\u2705 Config ready.")
print(f"   Model         : {MODEL_NAME}")
print(f"   Streaming     : {STREAM}")
print(f"   Summarize every: {SUMMARY_EVERY} turns")

---
## 📝 Cell 3 — Prompt Templates

In [ ]:
# ============================================================
# CELL 3: Prompt Templates
# ============================================================
# Two templates:
#   1. TRAVEL_SYSTEM_PROMPT  — defines the travel agent persona
#      and the strict rule to refuse non-travel questions.
#   2. SUMMARY_PROMPT        — used internally every 5 turns
#      to compress chat history into a short summary.

from langchain_core.prompts import ChatPromptTemplate, MessagesPlaceholder
from langchain_core.messages import SystemMessage, HumanMessage, AIMessage


# ── 1. Main travel agent system prompt ───────────────────
# The guard rule is embedded directly in the system message.
# The LLM is instructed to respond with the exact refusal string
# for anything outside travel — no apologies, no elaboration.
TRAVEL_SYSTEM_PROMPT = """\
You are an expert Travel Agent AI with deep knowledge of:
  - International and domestic flights and airlines
  - Hotels, resorts, hostels, and accommodation options
  - Popular and off-the-beaten-path travel destinations
  - Visa requirements, travel documents, and entry rules
  - Trip planning, itineraries, and packing tips
  - Travel insurance, budgeting, and currency exchange
  - Local culture, food, transport, and safety tips

STRICT RULE:
If the user asks about ANYTHING that is not travel-related
(e.g. coding, math, cooking, news, medical advice, etc.),
you MUST respond with EXACTLY this phrase and nothing else:
  I can't help with it

Do NOT apologise. Do NOT explain. Do NOT add any other text.
Just output:  I can't help with it

For travel questions, be helpful, specific, and enthusiastic.
Use bullet points and structure when listing options.
"""


# ── 2. Summarization prompt ───────────────────────────────
# Used to compress old messages into a single summary block.
# The summary replaces the full history to keep context small.
SUMMARY_SYSTEM_PROMPT = """\
You are a conversation summarizer.
Summarize the following travel chat history into 3-5 concise bullet points.
Capture: destinations discussed, recommendations made, user preferences,
and any decisions or plans mentioned.
Be brief — this summary will replace the full history to save context space.
"""


# ── Main chat prompt template ─────────────────────────────
# MessagesPlaceholder injects the full message list (history + current)
# dynamically at inference time.
chat_prompt = ChatPromptTemplate.from_messages([
    ("system", TRAVEL_SYSTEM_PROMPT),
    MessagesPlaceholder(variable_name="messages"),
])


# ── Summarization prompt template ─────────────────────────
summary_prompt = ChatPromptTemplate.from_messages([
    ("system", SUMMARY_SYSTEM_PROMPT),
    ("human", "Chat history to summarize:\n\n{history}"),
])

print("\u2705 Prompt templates defined.")

---
## 🧠 Cell 4 — Memory Manager

In [ ]:
# ============================================================
# CELL 4: Memory Manager
# ============================================================
# All memory is stored in a plain Python list: `chat_history`
# Each item is a LangChain message object:
#   HumanMessage(content="...")  — user turn
#   AIMessage(content="...")     — assistant turn
#
# How summarization works:
#   - turn_count tracks completed user+AI pairs
#   - Every SUMMARY_EVERY turns, the full history is sent
#     to the LLM with the summary prompt
#   - The returned summary replaces all old messages
#     with a single SystemMessage containing the digest
#   - This keeps the context window small for long sessions

from langchain_google_genai import ChatGoogleGenerativeAI
from langchain_core.messages import SystemMessage, HumanMessage, AIMessage


# ── Initialise the LLM ───────────────────────────────────
llm = ChatGoogleGenerativeAI(
    model=MODEL_NAME,
    temperature=TEMPERATURE,
)


# ── Memory state ─────────────────────────────────────────
chat_history: list = []   # stores all HumanMessage / AIMessage objects
turn_count:   int  = 0    # counts completed conversation turns


# ── Build the chains ─────────────────────────────────────
chat_chain    = chat_prompt    | llm   # main conversation chain
summary_chain = summary_prompt | llm   # summarization chain


def format_history_for_summary(messages: list) -> str:
    """
    Convert the message list to a plain text string
    so the summarization LLM can read it easily.
    """
    lines = []
    for msg in messages:
        if isinstance(msg, HumanMessage):
            lines.append(f"User: {msg.content}")
        elif isinstance(msg, AIMessage):
            lines.append(f"Assistant: {msg.content}")
        elif isinstance(msg, SystemMessage):
            lines.append(f"[Summary]: {msg.content}")
    return "\n".join(lines)


def summarize_and_compress() -> None:
    """
    Summarize the current chat_history and replace it
    with a single SystemMessage containing the digest.

    Called automatically every SUMMARY_EVERY turns.
    The summary is injected as a SystemMessage so the LLM
    knows it represents prior context, not a new user request.
    """
    global chat_history

    if not chat_history:
        return

    print("\n\U0001f4cb [Summarizing chat history...]")

    # Format history as readable text for the summary LLM
    history_text = format_history_for_summary(chat_history)

    # Call the summarization chain (not streamed — internal operation)
    summary_response = summary_chain.invoke({"history": history_text})
    summary_text     = summary_response.content

    # Replace the entire history with one compact SystemMessage
    # This keeps the prompt short while preserving context
    chat_history = [
        SystemMessage(content=f"Previous conversation summary:\n{summary_text}")
    ]

    print(f"\u2705 History compressed. Summary:\n")
    print(f"   {summary_text}\n")


def add_to_memory(role: str, content: str) -> None:
    """
    Append a message to chat_history.
    role: 'human' or 'ai'
    """
    if role == "human":
        chat_history.append(HumanMessage(content=content))
    elif role == "ai":
        chat_history.append(AIMessage(content=content))


def reset_memory() -> None:
    """Clear all history and reset turn counter."""
    global chat_history, turn_count
    chat_history = []
    turn_count   = 0
    print("\u2705 Memory cleared.")


print("\u2705 Memory manager ready.")
print(f"   Summarization trigger: every {SUMMARY_EVERY} turns")

---
## 🚀 Cell 5 — Streaming Chat Engine

In [ ]:
# ============================================================
# CELL 5: Streaming Chat Engine
# ============================================================
# chat() is the main entry point for every user message.
#
# STREAMING:
#   LangChain's .stream() yields chunks as the LLM generates.
#   Each chunk.content fragment is printed immediately with
#   end="" and flush=True to simulate real-time output.
#   The full response is assembled by concatenating all chunks.
#
# MEMORY FLOW:
#   1. Add user message to chat_history
#   2. Build prompt = system + chat_history (includes new message)
#   3. Stream LLM response token by token
#   4. Add AI response to chat_history
#   5. Increment turn_count
#   6. If turn_count % SUMMARY_EVERY == 0 → summarize & compress

def chat(user_input: str) -> str:
    """
    Process one user message and return the assistant's response.

    Steps:
    1. Add user message to memory
    2. Stream response from Gemini token-by-token
    3. Add AI response to memory
    4. Trigger summarization every SUMMARY_EVERY turns

    Returns the full assistant response as a string.
    """
    global turn_count

    # ── Step 1: Store user message in memory ─────────────
    add_to_memory("human", user_input)

    # ── Step 2: Stream the response ──────────────────────
    # chat_history now includes the new human message,
    # so the LLM sees the full conversation context.
    print("\n\U0001f916 Travel Agent: ", end="", flush=True)

    full_response = ""

    # .stream() returns an iterator of chunk objects.
    # Each chunk.content is a small string fragment (1-5 tokens).
    # Printing immediately gives the streaming effect.
    for chunk in chat_chain.stream({"messages": chat_history}):
        fragment = chunk.content
        print(fragment, end="", flush=True)  # live token output
        full_response += fragment

    print()  # newline after streaming completes

    # ── Step 3: Store AI response in memory ──────────────
    add_to_memory("ai", full_response)

    # ── Step 4: Increment turn and check summarization ───
    # A "turn" = one complete user+AI exchange.
    turn_count += 1

    if turn_count % SUMMARY_EVERY == 0:
        # Compress history every SUMMARY_EVERY turns.
        # This keeps the context window manageable for long chats.
        summarize_and_compress()

    return full_response


def show_memory_status() -> None:
    """Print current memory state — useful for debugging."""
    print(f"\n\U0001f9e0 Memory Status:")
    print(f"   Turn count   : {turn_count}")
    print(f"   Messages in memory: {len(chat_history)}")
    print(f"   Next summary at turn: "
          f"{((turn_count // SUMMARY_EVERY) + 1) * SUMMARY_EVERY}")
    for i, msg in enumerate(chat_history):
        role    = type(msg).__name__.replace("Message", "")
        preview = msg.content[:60].replace("\n", " ")
        print(f"   [{i}] {role}: {preview}..." if len(msg.content) > 60
              else f"   [{i}] {role}: {preview}")


print("\u2705 Streaming chat engine ready.")
print("   Run Cell 6 to start chatting.")

---
## 🎮 Cell 6 — Interactive Chatbot Demo

In [ ]:
# ============================================================
# CELL 6: Interactive Chatbot Demo
# ============================================================
# Commands during chat:
#   'quit' or 'exit'  → stop the chatbot
#   'memory'          → show current memory state
#   'reset'           → clear all history and start fresh
#   'history'         → print full conversation so far
#
# The chatbot will:
#   ✅ Answer travel questions with streaming output
#   ❌ Refuse non-travel questions with "I can't help with it"
#   📋 Auto-summarize after every 5 turns

print("\n" + "=" * 60)
print("  \u2708\ufe0f  TRAVEL AGENT CHATBOT  \u2708\ufe0f")
print("=" * 60)
print("  Ask me anything about travel: flights, hotels,")
print("  destinations, itineraries, visas, tips & more!")
print("  Type 'quit' to exit | 'memory' | 'reset' | 'history'")
print("=" * 60 + "\n")

# Reset memory at the start of each demo run
reset_memory()

while True:
    try:
        user_input = input("\U0001f9f3 You: ").strip()
    except (EOFError, KeyboardInterrupt):
        print("\n\U0001f44b Goodbye! Safe travels!")
        break

    # ── Built-in commands ─────────────────────────────────
    if not user_input:
        continue

    if user_input.lower() in ("quit", "exit", "q"):
        print("\U0001f44b Goodbye! Safe travels!")
        break

    if user_input.lower() == "memory":
        show_memory_status()
        continue

    if user_input.lower() == "reset":
        reset_memory()
        print("\U0001f504 Chat history cleared. Starting fresh!")
        continue

    if user_input.lower() == "history":
        print("\n\U0001f4dc Full conversation history:")
        if not chat_history:
            print("   (empty)")
        for msg in chat_history:
            role = type(msg).__name__.replace("Message", "")
            print(f"\n  [{role}]\n  {msg.content}")
        print()
        continue

    # ── Main chat call ────────────────────────────────────
    # All streaming, memory, and summarization logic
    # is handled inside chat().
    chat(user_input)

---
## 🧪 Cell 7 — Automated Test (Optional)
Runs 7 scripted messages to verify travel guard, streaming, and summarization.

In [ ]:
# ============================================================
# CELL 7: Automated Test — verifies all 3 core behaviours
# ============================================================
# Sends 7 scripted messages:
#   Turns 1-5 : travel questions  → streaming responses
#   Turn 5    : triggers auto-summarization
#   Turn 6    : non-travel question → 'I can't help with it'
#   Turn 7    : travel again → normal response resumes

import time

TEST_MESSAGES = [
    # ── Travel questions (turns 1–5) ─────────────────────
    "What are the top 3 destinations in Southeast Asia for first-time travellers?",
    "I have a budget of $2000 for 10 days in Japan. Is that enough?",
    "What documents do I need to visit Schengen countries from the US?",
    "Can you suggest a 5-day itinerary for Bali including temples and beaches?",
    "What is the best time of year to visit Patagonia?",
    # ── Non-travel question (turn 6) — should be refused ─
    "Can you write me a Python function to sort a list?",
    # ── Back to travel (turn 7) ──────────────────────────
    "What airlines fly direct from New York to Tokyo?",
]

print("\U0001f9ea Starting automated test...\n")
print("Expected: turns 1-5 → travel answers (streaming)")
print("          turn 5   → summarization triggered")
print("          turn 6   → \"I can't help with it\"")
print("          turn 7   → normal travel answer")
print("-" * 60)

reset_memory()

for i, msg in enumerate(TEST_MESSAGES, 1):
    print(f"\n\U0001f9f3 [Test {i}/7] You: {msg}")
    response = chat(msg)
    # Verify turn 6 refusal
    if i == 6:
        if "can't help" in response.lower():
            print("   \u2705 PASS: Non-travel question correctly refused")
        else:
            print("   \u274c FAIL: Should have refused this non-travel question")
    time.sleep(2)  # polite delay between API calls

print("\n" + "=" * 60)
print("\u2705 Automated test complete.")
show_memory_status()